# BiLSTM Speech Enhancement - Colab training

**Before running:** set the runtime to **GPU** (Runtime -> Change runtime type -> T4 GPU).

One-time dataset prep on your PC (see `scripts/prepare_dataset.md`):
```powershell
cd <repo>
tar -cvf voicebank_demand.tar -C speech clean_trainset_wav noisy_trainset_wav clean_testset_wav noisy_testset_wav
```
then upload `voicebank_demand.tar` (~2.6 GB) to the root of your Google Drive (`MyDrive/`).

In [ ]:
# 1. Clone the rebuild branch
!git clone --branch bilstm-rebuild https://github.com/AsimShareef/BiLSTM-Speech-Enhancement-System.git repo
%cd repo
!git log --oneline -3

In [ ]:
# 2. Dependencies (Colab already has tensorflow + numpy/scipy/librosa)
!pip -q install pystoi pesq
import tensorflow as tf
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 3. Pull the dataset from Drive and extract to ./speech
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p speech
!tar -xf /content/drive/MyDrive/voicebank_demand.tar -C speech
!ls speech && echo '---' && ls speech/clean_trainset_wav | wc -l

In [ ]:
# 4. Sanity-check the streaming dataset (no training yet)
!python src/dataset.py

In [ ]:
# 5. Train on the full corpus (early stopping usually halts well before 40 epochs)
# --checkpoint-dir points at Drive: if the Colab runtime disconnects mid-run,
# the best weights / history so far are already saved there, not lost with the VM.
!python src/train.py --epochs 40 --batch 128 \
    --checkpoint-dir /content/drive/MyDrive/bilstm_se_out/checkpoints

In [ ]:
# 6. Evaluate on all 824 test files: BiLSTM vs spectral-subtraction vs noisy
# Uses the local save from cell 5; if that VM session was interrupted and
# restarted, point --model at the Drive copy instead:
#   --model /content/drive/MyDrive/bilstm_se_out/checkpoints/bilstm_enhancer.keras
!python src/evaluate.py --model bilstm_enhancer.keras
print(open('results/metrics.md').read())

In [ ]:
# 7. Save the remaining artefacts back to Drive (model + checkpoints already
# went there live during training via --checkpoint-dir; this just adds results/)
!cp -r results /content/drive/MyDrive/bilstm_se_out/
print('done - MyDrive/bilstm_se_out/ now has: checkpoints/ (model, history.csv, history.json) + results/')
print('download results/metrics.md and checkpoints/history.csv from Drive and share them back')